<a href="https://colab.research.google.com/github/yangyuwang/wikiart_metadata/blob/main/wikipedia_demographic_extract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load wikipedia

In [2]:
from google.colab import drive
drive.mount('/content/drive/')


Mounted at /content/drive/


In [9]:
import os
import re
from tqdm import tqdm

artist_wikipedia_text_example = []

path = '/content/drive/My Drive/artist_wikipedia_content/'

target_name = [re.sub(r".txt", "", target) for target in os.listdir(path)]

for file in tqdm(os.listdir(path)):
  with open(path + file, "r") as f:
    artist_wikipedia_text_example.append(re.sub("\n", "", f.read()))

100%|██████████| 2752/2752 [00:08<00:00, 315.85it/s]


# OPENAI settings

In [10]:
pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.70.0
    Uninstalling openai-1.70.0:
      Successfully uninstalled openai-1.70.0


In [11]:
import getpass
OPENAI_API_KEY = getpass.getpass()

··········


# Extraction

In [12]:
# Few-shot example using Agostino Carracci's information.
few_shot_example = """
                    Example1:
                    Text: "Agostino Carracci (or Caracci; 16 August 1557 – 22 March 1602) was an Italian painter born in Bologna. He resided in Bologna in 1557 and later moved to Venice in 1590. He visited Rome in 1580 and had interactions with Prospero Fontana (Teacher, 1575) and Bartolomeo Passarotti (Mentor, 1585)."
                    Chain-of-thought:
                    1. Educational Level: Agostino was trained by established artists, so his educational level is "Taught by other artists".
                    2. Gender: The text indicates he is male.
                    3. Residences: From the text, we extract that he was born in Bologna in 1557 and later moved to Venice in 1590. Thus, the residences are [{ "city": "Bologna", "year": 1557 }, { "city": "Venice", "year": 1590 }].
                    4. Visits: The text mentions that he visited Rome in 1580, so we record [{ "city": "Rome", "year": 1580 }].
                    5. Interactions: The text notes interactions with Prospero Fontana as a Teacher in 1575 and Bartolomeo Passarotti as a Mentor in 1585, so these are recorded as [{ "name": "Prospero Fontana", "relation": "Teacher", "year": 1575 }, { "name": "Bartolomeo Passarotti", "relation": "Mentor", "year": 1585 }].
                    Final JSON Output:
                    {
                      "Educational Level": "Taught by other artists",
                      "Gender": "Male",
                      "Residences": [{ "city": "Bologna", "year": 1557 }, { "city": "Venice", "year": 1590 }],
                      "Visits": [{ "city": "Rome", "year": 1580 }],
                      "Interactions": [{ "name": "Prospero Fontana", "relation": "Teacher", "year": 1575 }, { "name": "Bartolomeo Passarotti", "relation": "Mentor", "year": 1585 }]
                    }

                    Example2:
                    Text: "Banksy is a renowned street artist, but his true identity remains a mystery. Minimal reliable information is available regarding his background. There are no clear details about his education, gender, places of residence, cities he may have visited, or any documented interactions with other artists."
                    Chain-of-thought:
                    1. Educational Level: The text provides no details about Banksy's education, so the Educational Level is unknown.
                    2. Gender: The text does not clarify Banksy's gender, so Gender is unknown.
                    3. Residences: There is no available information on any cities or years of residence, hence the Residences list is empty.
                    4. Visits: There is no mention of any visits to specific cities or associated years, so the Visits list is empty.
                    5. Interactions: The text does not document any interactions with other notable individuals, so the Interactions list is empty.
                    Final JSON Output:
                    {
                      "Educational Level": "Unknown",
                      "Gender": "Unknown",
                      "Residences": [],
                      "Visits": [],
                      "Interactions": []
                    }

                    Example3:
                    Text: "Frida Kahlo, the celebrated Mexican painter, spent much of her life in Mexico City. She also visited Paris and engaged actively with the art community there. Notably, she interacted with Diego Rivera (Partner) and Tarsila do Amaral (Fellow Artist), although the biography does not provide any dates for these interactions or her movements."
                    Chain-of-thought:
                    1. Educational Level: The text does not mention any details regarding her formal or informal education, so her Educational Level is unknown.
                    2. Gender: The text does not explicitly state her gender, so Gender is unknown.
                    3. Residences: The text indicates she resided in Mexico City, but no year is provided. Therefore, we record the residence as [{ "city": "Mexico City", "year": "Unknown" }].
                    4. Visits: The text mentions that she visited Paris, but without specifying a year. Hence, we record this as [{ "city": "Paris", "year": "Unknown" }].
                    5. Interactions: The text notes that she interacted with Diego Rivera (Partner) and Tarsila do Amaral (Fellow Artist), but it does not mention the years of these interactions. Thus, we record the interactions as [{ "name": "Diego Rivera", "relation": "Partner", "year": "Unknown" }, { "name": "Tarsila do Amaral", "relation": "Fellow Artist", "year": "Unknown" }].
                    Final JSON Output:
                    {
                      "Educational Level": "Unknown",
                      "Gender": "Unknown",
                      "Residences": [{ "city": "Mexico City", "year": "Unknown" }],
                      "Visits": [{ "city": "Paris", "year": "Unknown" }],
                      "Interactions": [
                        { "name": "Diego Rivera", "relation": "Partner", "year": "Unknown" },
                        { "name": "Tarsila do Amaral", "relation": "Fellow Artist", "year": "Unknown" }] }
                    ---
                    """

# Build the full prompt including the few-shot example.
prompt = f"""
          You are an expert text analyst. Below is a few-shot example showing how to extract demographic attributes from a Wikipedia-style text about an artist using a chain-of-thought approach.

          {few_shot_example}

          Now, analyze the following text and extract the following attributes:
          - Educational Level: Choose one from ["No formal education", "Elementary level", "Middle school level", "High school level", "College level", "Master level", "PhD level", "Taught by other artists"].
          - Gender
          - Residences: Provide a list of dictionaries where each dictionary contains the keys "city" and "year", representing the cities where the artist resided and the associated year.
          - Visits: Provide a list of dictionaries where each dictionary contains the keys "city" and "year", representing the cities the artist visited and the associated year.
          - Interactions: Provide a list of dictionaries, with each dictionary containing "name", "relation", and "year", representing the interactions the artist had (for example, mentors or notable influences).

          Please follow these instructions:
          1. First, provide your chain-of-thought reasoning step-by-step.
          2. Then, based on your reasoning, output a final answer in JSON format with the keys "Educational Level", "Gender", "Residences", "Visits", and "Interactions".
          3. Output only a valid JSON object (without any additional text, code fences, or markdown) as the final answer.

          """

In [13]:
message = [{"role": "system", "content": prompt}]

In [14]:
import re
import json

def extract_last_balanced_json(text):
    """
    Scans the text and returns the last balanced JSON block found.
    """
    blocks = []
    count = 0
    block_start = None
    # Iterate over all characters
    for i, ch in enumerate(text):
        if ch == '{':
            if count == 0:  # start of a new block
                block_start = i
            count += 1
        elif ch == '}':
            if count:
                count -= 1
                if count == 0 and block_start is not None:
                    blocks.append((block_start, i+1))
    if blocks:
        last_start, last_end = blocks[-1]
        return text[last_start:last_end]
    return None

def extract_json(answer):
    # Try to extract a JSON block from a Markdown code fence first.
    json_match = re.search(r"```json\s*(\{.*\})\s*```", answer, re.DOTALL)

    if json_match:
        json_str = json_match.group(1)
    else:
        # Fallback: use the extraction of the last balanced JSON block
        json_str = extract_last_balanced_json(answer)
        if not json_str:
            # If no JSON block is found, return an empty result.
            return {}, ""

    try:
        result = json.loads(json_str)
        return result, json_str
    except json.JSONDecodeError:
        print("JSON decode error. Full response saved")
        return {}, json_str

In [15]:
import openai
import json
import pandas as pd

# Set your OpenAI API key
openai.api_key = OPENAI_API_KEY

def extract_demographics_few_shot(text):

    message = [{"role": "system", "content": prompt}]
    message.append({"role": "user", "content": f"""
                                                Text:
                                                \"\"\"{text}\"\"\"

                                                Final JSON Output:
                                                """})

    # Call the GPT-3.5 model
    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=message,
        temperature=0
    )

    # Get the content of the reply
    answer = response["choices"][0]["message"]["content"]

    return answer


In [70]:
# Example

# Wikipedia texts for Agostino Carracci and Agnolo Bronzino
text_carracci = """
Agostino Carracci (or Caracci; Italian pronunciation: [aɡoˈstiːno karˈrattʃi]; 16 August 1557 – 22 March 1602) was an Italian painter, printmaker, tapestry designer, and art teacher. He was, together with his brother, Annibale Carracci, and cousin, Ludovico Carracci, one of the founders of the Accademia degli Incamminati (Academy of the Progressives) in Bologna. Intended to devise alternatives to the Mannerist style favored in the preceding decades, this teaching academy helped propel painters of the School of Bologna to prominence.

Agostino Carracci was born in Bologna as the son of a tailor. He was the elder brother of Annibale Carracci and the cousin of Ludovico Carracci. He initially trained as a goldsmith. He later studied painting, first with Prospero Fontana, who had been Lodovico's master, and later with Bartolomeo Passarotti. He traveled to Parma to study the works of Correggio. Accompanied by his brother Annibale, he spent a long time in Venice, where he trained as an engraver under the renowned Cornelis Cort.
"""

text_bronzino = """
Agnolo di Cosimo (Italian: [ˈaɲɲolo di ˈkɔːzimo]; 17 November 1503 – 23 November 1572), usually known as Bronzino or Agnolo Bronzino, was an Italian Mannerist painter from Florence. His sobriquet, Bronzino, may refer to his relatively dark skin or reddish hair.
He lived all his life in Florence, and from his late 30s was kept busy as the court painter of Cosimo I de' Medici, Grand Duke of Tuscany. He was mainly a portraitist, but also painted many religious subjects and a few allegorical subjects. He trained with Pontormo, the leading Florentine painter of the first generation of Mannerism, and his style was greatly influenced by him. He was apprenticed at 14 under Pontormo and later studied with Raffaellino del Garbo.
"""

# Process each text to extract demographics using the few-shot approach
results = {}
results["Agostino Carracci"], text1 = extract_json(extract_demographics_few_shot(text_carracci))
results["Agnolo Bronzino"], text2 = extract_json(extract_demographics_few_shot(text_bronzino))


In [71]:
results

{'Agostino Carracci': {'Educational Level': 'Taught by other artists',
  'Gender': 'Male',
  'Residences': [{'city': 'Bologna', 'year': 'Unknown'},
   {'city': 'Venice', 'year': 'Unknown'}],
  'Visits': [{'city': 'Parma', 'year': 'Unknown'}],
  'Interactions': [{'name': 'Prospero Fontana',
    'relation': 'Teacher',
    'year': 'Unknown'},
   {'name': 'Bartolomeo Passarotti', 'relation': 'Teacher', 'year': 'Unknown'},
   {'name': 'Cornelis Cort', 'relation': 'Trainer', 'year': 'Unknown'}]},
 'Agnolo Bronzino': {'Educational Level': 'Taught by other artists',
  'Gender': 'Male',
  'Residences': [{'city': 'Florence', 'year': 'Unknown'}],
  'Visits': [],
  'Interactions': [{'name': 'Pontormo',
    'relation': 'Teacher',
    'year': 'Unknown'},
   {'name': 'Raffaellino del Garbo',
    'relation': 'Teacher',
    'year': 'Unknown'}]}}

In [16]:
def summarize_text(text, max_words=200):
    prompt = f"Summarize the following text in {max_words} words while keeping all relevant biographical and demographic details:\n\n{text}"

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )

    return response["choices"][0]["message"]["content"]

In [ ]:
import json
import pandas as pd

# Load the JSON data from the file
with open('/content/drive/My Drive/artist_data/demographic.json', 'r') as f:
    demographic = json.load(f)

with open('/content/drive/My Drive/artist_data/text_list.json', 'w') as f:
    text_list = json.load(f)

In [17]:
demographic = {}
text_list = []


In [18]:
n = 0

for k, i in demographic.items():
    if n == 5:
       break
    print(k, i)
    n += 1

In [20]:
import json

for i in tqdm(range(len(target_name))):
    if target_name[i] not in demographic.keys():
        try:
            atrributes, str_json = extract_json(extract_demographics_few_shot(artist_wikipedia_text_example[i]))
        except Exception as e:
            print("Too long... Summarize first")
            atrributes, str_json = extract_json(extract_demographics_few_shot(summarize_text(artist_wikipedia_text_example[i])))
        text_list.append(str_json)
        demographic[target_name[i]] = atrributes

    if i % 100 == 0:
        # Save demographic data to a JSON file
        with open('/content/drive/My Drive/artist_data/demographic.json', 'w') as f:
          json.dump(demographic, f, indent=2)

        # Save text_list data to a JSON file
        with open('/content/drive/My Drive/artist_data/text_list.json', 'w') as f:
          json.dump(text_list, f, indent=2)


 70%|██████▉   | 1925/2752 [2:37:13<1:42:36,  7.44s/it]

Too long... Summarize first


100%|██████████| 2752/2752 [3:44:36<00:00,  4.90s/it]


In [23]:
with open('/content/drive/My Drive/artist_data/demographic.json', 'w') as f:
  json.dump(demographic, f, indent=2)

with open('/content/drive/My Drive/artist_data/text_list.json', 'w') as f:
  json.dump(text_list, f, indent=2)

# Data Cleaning and EDA

In [51]:
import pandas as pd

file_id = '1Wa_MueOpfdXZveRS5J-eMLyaBQ-SeFtN'
uuid = '6139ffcf-19c1-46b4-b013-a33b11655f4d'
at = 'AIrpjvPTjK6nk2lXSIJKc8rBEEqH:1739041451599'

download_url = f'https://drive.usercontent.google.com/download?id={file_id}&export=download&authuser=0&confirm=t&uuid={uuid}&at={at}'

df = pd.read_csv(download_url)

df.head()

<ipython-input-51-67d5d9e385a8>:9: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(download_url)


,Artist_name,Artwork.Name,Year,Link,url,Create.Date,Create.Location,Date,image_url,Style,...,Wikipedia,Painting.School,Official.site,Friends.and.Co.workers,Family.and.Relatives,Pupils,Teachers,Art.institution,Artwork_ID,Style_Category
0,3d,No Great Crime,1983,https://www.wikiart.org/en/3d/no-great-crime-1983,https://www.wikiart.org/en/3d/no-great-crime-1983,1983.0,NaN,1983,https://uploads2.wikiart.org/images/3d/no-grea...,Street art,...,https://en.wikipedia.org/wiki/Robert_Del_Naja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Street & Urban Art
1,3d,3D,1984,https://www.wikiart.org/en/3d/3d-1984,https://www.wikiart.org/en/3d/3d-1984,1984.0,NaN,1984,https://uploads7.wikiart.org/images/3d/3d-1984...,Street art,...,https://en.wikipedia.org/wiki/Robert_Del_Naja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,Street & Urban Art
2,3d,Wild Bunch,1984,https://www.wikiart.org/en/3d/wild-bunch-1984,https://www.wikiart.org/en/3d/wild-bunch-1984,1984.0,NaN,1984,https://uploads6.wikiart.org/images/3d/wild-bu...,Street art,...,https://en.wikipedia.org/wiki/Robert_Del_Naja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,Street & Urban Art
3,3d,Serious Art,1986,https://www.wikiart.org/en/3d/serious-art-1986,https://www.wikiart.org/en/3d/serious-art-1986,1986.0,NaN,1986,https://uploads8.wikiart.org/images/3d/serious...,Street art,...,https://en.wikipedia.org/wiki/Robert_Del_Naja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,Street & Urban Art
4,3d,"Robert De Niro, Taxi Driver",1988,https://www.wikiart.org/en/3d/robert-de-niro-t...,https://www.wikiart.org/en/3d/robert-de-niro-t...,1988.0,NaN,1988,https://uploads5.wikiart.org/images/3d/robert-...,Street art,...,https://en.wikipedia.org/wiki/Robert_Del_Naja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,Street & Urban Art


In [54]:
df.columns

Index(['Artist_name', 'Artwork.Name', 'Year', 'Link', 'url', 'Create.Date',
       'Create.Location', 'Date', 'image_url', 'Style', 'Genre.x', 'Media',
       'tags', 'Location', 'Series', 'Period', 'Theme', 'Share', 'image_n',
       'name', 'Birth.Date', 'Death.Date', 'Birth.place', 'Death.place',
       'Nationality', 'Art.Movement', 'Genre.y', 'Field', 'Influenced.by',
       'Influenced.on', 'Wikipedia', 'Painting.School', 'Official.site',
       'Friends.and.Co.workers', 'Family.and.Relatives', 'Pupils', 'Teachers',
       'Art.institution', 'Artwork_ID', 'Style_Category'],
      dtype='object')

In [57]:
name_to_artist_name = dict(zip(df["name"], df["Artist_name"]))

In [65]:
def normalize_name(name):
  """Converts a name like "Yangyu Wang" to "yangyu-wang".

  Args:
    name: The input name string.

  Returns:
    The normalized name string.
  """
  return name.lower().replace(" ", "-")


In [68]:
for name, data in demographic.items():
    for res in data["Interactions"]:
        res['name'] = name_to_artist_name.get(res['name'], normalize_name(res['name']))
    for res in data["Residences"]:
        res['city'] = normalize_name(res['city'])
    for res in data["Visits"]:
        res['city'] = normalize_name(res['city'])

In [70]:
with open('/content/drive/My Drive/artist_data/demographic_namemapped.json', 'w') as f:
  json.dump(demographic, f, indent=2)

In [72]:

import json
import pandas as pd

# Load the JSON data from the file
with open('/content/drive/My Drive/artist_data/demographic_namemapped.json', 'r') as f:
    demographic = json.load(f)

# Convert the JSON data to a Pandas DataFrame
df = pd.DataFrame.from_dict(demographic, orient='index')

# Save the DataFrame as a CSV file
df.to_csv('/content/drive/My Drive/artist_data/demographic.csv')

In [73]:
residence_city_list = []

for name, data in demographic.items():
    cities = [res['city'] for res in data["Residences"]]
    residence_city_list += cities

In [74]:
visit_city_list = []

for name, data in demographic.items():
    cities = [res['city'] for res in data["Visits"]]
    visit_city_list += cities

In [79]:
len(visit_city_list + residence_city_list)

16809

In [76]:
interaction_list = []

for name, data in demographic.items():
    ints = [res['name'] for res in data["Interactions"]]
    interaction_list += ints

In [78]:
len(interaction_list)

12917